# Data Loading and Initial Inspection

This notebook loads the project's sample dataset and performs the first checks needed before analysis or model training.

**Dataset:** `sample_data/student-mat.csv`

> Run the cells from top to bottom. The path setup works whether Jupyter is opened from the project root or from the `notebooks` folder.

In [1]:
# All imports belong at the top of the notebook
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Find the project root so the notebook works from either common working directory.
possible_roots = [Path.cwd(), Path.cwd().parent]
project_root = next(
    (path for path in possible_roots if (path / "modules").is_dir()),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the project "
        "root or its notebooks folder."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules.data_loader import load_csv, validate_dataset

print("Project root:", project_root)

C:\Users\PMLS\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
C:\Users\PMLS\AppData\Roaming\Python\Python313\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Project root: c:\Users\PMLS\Documents\Coding\System-7-Internship\AutoML-Studio


## 1. Load the dataset

The project's `load_csv()` helper automatically detects the CSV delimiter.

In [2]:
dataset_path = project_root / "sample_data" / "student-mat.csv"

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path}\n"
        "Make sure student-mat.csv is inside the project's sample_data folder."
    )

df = load_csv(dataset_path)
validate_dataset(df)

print("Dataset loaded successfully!")
print("File:", dataset_path.name)
df.head()

Dataset loaded successfully!
File: student-mat.csv


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


## 2. Dataset dimensions and columns

In [3]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print("\nColumn names:")
print(df.columns.tolist())

Rows: 395
Columns: 33

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


## 3. Data types and structure

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      395 non-null    str  
 1   sex         395 non-null    str  
 2   age         395 non-null    int64
 3   address     395 non-null    str  
 4   famsize     395 non-null    str  
 5   Pstatus     395 non-null    str  
 6   Medu        395 non-null    int64
 7   Fedu        395 non-null    int64
 8   Mjob        395 non-null    str  
 9   Fjob        395 non-null    str  
 10  reason      395 non-null    str  
 11  guardian    395 non-null    str  
 12  traveltime  395 non-null    int64
 13  studytime   395 non-null    int64
 14  failures    395 non-null    int64
 15  schoolsup   395 non-null    str  
 16  famsup      395 non-null    str  
 17  paid        395 non-null    str  
 18  activities  395 non-null    str  
 19  nursery     395 non-null    str  
 20  higher      395 non-null    str  
 21  inte

In [5]:
# Review the data type of every column
display(df.dtypes.to_frame(name="Data Type"))

,Data Type
school,str
sex,str
age,int64
address,str
famsize,str
Pstatus,str
Medu,int64
Fedu,int64
Mjob,str
Fjob,str


## 4. Data quality checks

In [6]:
# Missing values in each column
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary.sort_values("Missing Values", ascending=False)

,Missing Values,Missing Percentage
school,0,0.0
sex,0,0.0
age,0,0.0
address,0,0.0
famsize,0,0.0
Pstatus,0,0.0
Medu,0,0.0
Fedu,0,0.0
Mjob,0,0.0
Fjob,0,0.0


In [7]:
# Duplicate rows
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [8]:
# A compact dataset quality summary
quality_summary = pd.DataFrame({
    "Check": ["Rows", "Columns", "Missing cells", "Duplicate rows"],
    "Result": [
        len(df),
        df.shape[1],
        int(df.isna().sum().sum()),
        int(duplicate_count)
    ]
})

display(quality_summary)

,Check,Result
0,Rows,395
1,Columns,33
2,Missing cells,0
3,Duplicate rows,0


## 5. Identify numerical and categorical columns

In [9]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:", numeric_columns)
print("\nCategorical columns:", categorical_columns)

Numerical columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


## 6. Basic statistical summaries

In [10]:
# Summary statistics for numerical columns
df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
age,395.0,16.696203,1.276043,15.0,16.0,17.0,18.0,22.0
Medu,395.0,2.749367,1.094735,0.0,2.0,3.0,4.0,4.0
Fedu,395.0,2.521519,1.088201,0.0,2.0,2.0,3.0,4.0
traveltime,395.0,1.448101,0.697505,1.0,1.0,1.0,2.0,4.0
studytime,395.0,2.035443,0.839240,1.0,1.0,2.0,2.0,4.0
failures,395.0,0.334177,0.743651,0.0,0.0,0.0,0.0,3.0
famrel,395.0,3.944304,0.896659,1.0,4.0,4.0,5.0,5.0
freetime,395.0,3.235443,0.998862,1.0,3.0,3.0,4.0,5.0
goout,395.0,3.108861,1.113278,1.0,2.0,3.0,4.0,5.0
Dalc,395.0,1.481013,0.890741,1.0,1.0,1.0,2.0,5.0


In [11]:
# Number of distinct values in each column
unique_summary = pd.DataFrame({
    "Unique Values": df.nunique(dropna=True),
    "Data Type": df.dtypes.astype(str)
})

unique_summary

,Unique Values,Data Type
school,2,str
sex,2,str
age,8,int64
address,2,str
famsize,2,str
Pstatus,2,str
Medu,5,int64
Fedu,5,int64
Mjob,5,str
Fjob,5,str


In [12]:
# Frequency counts for categorical columns
for column in categorical_columns:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False).head(10).to_frame("Count"))


school


,Count
school,
GP,349
MS,46



sex


,Count
sex,
F,208
M,187



address


,Count
address,
U,307
R,88


... (24 outputs left out)


internet


,Count
internet,
yes,329
no,66



romantic


,Count
romantic,
no,263
yes,132


## 7. Final validation

Confirm that the dataset passed the basic checks before moving to detailed exploratory data analysis and preprocessing.

In [13]:
validate_dataset(df)
print("Validation passed: the dataset is non-empty and has at least two columns.")
print("\nData loading and initial inspection are complete.")

Validation passed: the dataset is non-empty and has at least two columns.

Data loading and initial inspection are complete.
